In [89]:
import json
import os
import re
from datetime import datetime
from ollama import chat


In [90]:
import json
import pandas as pd

file_path = "physician_copilot_student_cases.json"

with open(file_path, "r") as f:
    data = json.load(f)

df = pd.json_normalize(data)
print(df.head())

  patient_id  age     sex  fasting                         conditions  \
0    LAB_001   29  female     True                                 []   
1    LAB_002   44    male    False                     [hypertension]   
2    LAB_003   63    male     True  [type 2 diabetes, hyperlipidemia]   
3    LAB_004   38  female     True                                 []   
4    LAB_005   51  female     True                          [obesity]   

                 medications              symptoms  labs.LDL  labs.HDL  \
0                         []             [fatigue]       118        62   
1               [amlodipine]                    []       154        38   
2  [metformin, atorvastatin]                    []        88        36   
3                         []  [nausea, dark urine]       102        54   
4                         []                    []       172        34   

   labs.Triglycerides  labs.A1c  labs.Creatinine  labs.eGFR  labs.ALT  \
0                  92       5.2            

In [91]:
LAB_RULES = {
    "LDL": {
        "units": "mg/dL",
        "normal": lambda v: v < 100,
        "categories": [
            (100, 129, "borderline high"),
            (130, 159, "high"),
            (160, float("inf"), "very high"),
        ],
    },
    "HDL": {
        "units": "mg/dL",
        "normal": lambda v: v >= 40,
        "categories": [
            (-float("inf"), 40, "low"),
        ],
    },
    "Triglycerides": {
        "units": "mg/dL",
        "normal": lambda v: v < 150,
        "categories": [
            (150, 199, "borderline high"),
            (200, 499, "high"),
            (500, float("inf"), "very high"),
        ],
    },
    "A1c": {
        "units": "%",
        "normal": lambda v: v < 5.7,
        "categories": [
            (5.7, 6.4, "prediabetes range"),
            (6.5, float("inf"), "diabetes range"),
        ],
    },
    "Creatinine": {
        "units": "mg/dL",
        "normal": lambda v: 0.6 <= v <= 1.3,
        "categories": [
            (-float("inf"), 0.6, "low"),
            (1.31, float("inf"), "high"),
        ],
    },
    "eGFR": {
        "units": "mL/min/1.73m²",
        "normal": lambda v: v >= 90,
        "categories": [
            (60, 89, "mildly decreased"),
            (45, 59, "moderately decreased"),
            (30, 44, "severely decreased"),
            (-float("inf"), 29, "kidney function very low"),
        ],
    },
    "ALT": {
        "units": "U/L",
        "normal": lambda v: v <= 40,
        "categories": [
            (41, 79, "mildly elevated"),
            (80, 149, "moderately elevated"),
            (150, float("inf"), "markedly elevated"),
        ],
    },
    "AST": {
        "units": "U/L",
        "normal": lambda v: v <= 40,
        "categories": [
            (41, 79, "mildly elevated"),
            (80, 149, "moderately elevated"),
            (150, float("inf"), "markedly elevated"),
        ],
    },
    "Vitamin_D": {
        "units": "ng/mL",
        "normal": lambda v: v >= 30,
        "categories": [
            (20, 29, "insufficient"),
            (0, 19, "deficient"),
        ],
    },
}


def range_check(lab, value):
    """
    Returns structured interpretation of a single lab:
    {
        lab, value, units, flagged, interpretation
    }
    """
    rule = LAB_RULES.get(lab)
    if rule is None:
        return {
            "lab": lab,
            "value": value,
            "units": "",
            "flagged": False,
            "interpretation": "unknown lab",
        }

    if rule["normal"](value):
        return {
            "lab": lab,
            "value": value,
            "units": rule["units"],
            "flagged": False,
            "interpretation": "normal",
        }

    for low, high, label in rule["categories"]:
        if low <= value <= high:
            return {
                "lab": lab,
                "value": value,
                "units": rule["units"],
                "flagged": True,
                "interpretation": label,
            }

    return {
        "lab": lab,
        "value": value,
        "units": rule["units"],
        "flagged": True,
        "interpretation": "abnormal",
    }


def severity_points(lab, value):
    """
    Internal helper used by severity_score.
    """
    if lab == "ALT":
        if value >= 150:
            return 4
        elif value >= 80:
            return 3
        elif value > 40:
            return 1

    if lab == "AST":
        if value >= 150:
            return 4
        elif value >= 80:
            return 3
        elif value > 40:
            return 1

    if lab == "A1c":
        if value >= 8.0:
            return 3
        elif value >= 6.5:
            return 2
        elif value >= 5.7:
            return 1

    if lab == "eGFR":
        if value < 30:
            return 4
        elif value < 45:
            return 3
        elif value < 60:
            return 2
        elif value < 90:
            return 1

    if lab == "Creatinine":
        if value > 1.5:
            return 2
        elif value > 1.3:
            return 1

    if lab == "Triglycerides":
        if value >= 500:
            return 4
        elif value >= 200:
            return 2
        elif value >= 150:
            return 1

    if lab == "LDL":
        if value >= 160:
            return 2
        elif value >= 130:
            return 1
        elif value >= 100:
            return 0.5

    if lab == "HDL":
        if value < 40:
            return 1

    if lab == "Vitamin_D":
        if value < 20:
            return 1
        elif value < 30:
            return 0.5

    return 0


def severity_score(labs):
    """
    Returns:
    {
        score: float,
        severity: str,
        rationale: list[str]
    }
    """
    total = 0
    rationale = []

    for lab, value in labs.items():
        pts = severity_points(lab, value)
        total += pts
        if pts > 0:
            rationale.append(f"{lab} contributed {pts} point(s)")

    
    urgent_trigger = (
        labs.get("ALT", 0) >= 150
        or labs.get("AST", 0) >= 150
        or labs.get("Triglycerides", 0) >= 500
        or labs.get("eGFR", 999) < 30
    )

    if urgent_trigger:
        severity = "Urgent follow-up"
        rationale.append("Urgent threshold triggered by at least one lab value")
    elif total >= 4:
        severity = "Follow-up recommended"
    else:
        severity = "Routine"

    return {
        "score": total,
        "severity": severity,
        "rationale": rationale,
    }


def prioritize_findings(flagged_labs):
    """
    Sort flagged labs by importance.
    """
    priority_order = {
        "ALT": 1,
        "AST": 2,
        "eGFR": 3,
        "Creatinine": 4,
        "A1c": 5,
        "Triglycerides": 6,
        "LDL": 7,
        "HDL": 8,
        "Vitamin_D": 9,
    }

    return sorted(
        flagged_labs,
        key=lambda x: (
            priority_order.get(x["lab"], 99),
            -abs(x["value"]) if isinstance(x["value"], (int, float)) else 0,
        ),
    )


def generate_followup_questions(labs, context):
    """
    Produce clarification questions for the physician or patient context.
    """
    questions = []

    if labs.get("ALT", 0) > 80 or labs.get("AST", 0) > 80:
        questions.append(
            "Has the patient had recent alcohol use, acetaminophen use, viral illness, supplements, or other possible liver stressors?"
        )
        if "dark urine" in context.get("symptoms", []) or "nausea" in context.get("symptoms", []):
            questions.append(
                "Given symptoms such as nausea or dark urine, should the patient be contacted sooner for clinical review?"
            )

    if labs.get("A1c", 0) >= 6.5:
        questions.append(
            "Does the patient have home glucose readings, recent dietary changes, or missed diabetes medication doses?"
        )

    if labs.get("eGFR", 999) < 60 or labs.get("Creatinine", 0) > 1.3:
        questions.append(
            "Should kidney function be repeated, and are there recent hydration changes, NSAID use, or other possible contributors?"
        )

    if labs.get("Triglycerides", 0) >= 200 and not context.get("fasting", False):
        questions.append(
            "Triglycerides are elevated on a non-fasting sample; should fasting labs be repeated?"
        )

    if labs.get("Vitamin_D", 999) < 20:
        questions.append(
            "Would you like to discuss vitamin D replacement or repeat testing at follow-up?"
        )

    return questions


In [92]:
def analyze_case(patient):
    labs = patient["labs"]

    lab_interpretations = []
    flagged = []

    for lab, value in labs.items():
        result = range_check(lab, value)
        lab_interpretations.append(result)
        if result["flagged"]:
            flagged.append(result)

    prioritized = prioritize_findings(flagged)
    severity = severity_score(labs)
    followup_questions = generate_followup_questions(labs, patient)

    audit = {
        "patient_id": patient["patient_id"],
        "abnormal_labs": prioritized,
        "severity": severity["severity"],
        "severity_score": severity["score"],
        "severity_rationale": severity["rationale"],
        "followup_questions": followup_questions,
    }

    return audit

In [93]:
def build_generation_prompt(patient, audit):
    return f"""
You are drafting a clinician-to-patient message for physician review.

Write the message as if the physician is speaking directly to the patient.

Important safety rules:
- Do not diagnose any condition.
- Do not say the patient definitely has a disease.
- Do not recommend medication changes.
- Do not provide dosage instructions.
- Do not make definitive clinical conclusions.
- Do not speculate beyond the provided lab results and context.
- Do not mention internal scoring, tools, or audit logic.
- Use cautious language such as:
  - "is outside the usual range"
  - "would be worth discussing"
  - "may need follow-up"
  - "we may want to repeat this test"
  - "I would like to review these results with you"

Style requirements:
- Keep the message clear, brief, and patient-friendly.
- Sound professional and calm.
- Focus only on the most important abnormal findings.
- Do not list every lab unless necessary.
- End with an appropriate next-step statement.

Severity category: {audit["severity"]}

Patient context:
- Patient ID: {patient["patient_id"]}
- Age: {patient["age"]}
- Sex: {patient["sex"]}
- Fasting: {patient["fasting"]}
- Conditions: {patient["conditions"]}
- Medications: {patient["medications"]}
- Symptoms: {patient["symptoms"]}

Lab results:
{json.dumps(patient["labs"], indent=2)}

Structured audit:
{json.dumps(audit, indent=2)}

Instructions by severity:
- If severity is "Routine":
  write a reassuring message, mention that a few results are slightly outside the usual range, and say they can be reviewed at follow-up.
- If severity is "Follow-up recommended":
  mention that several results would be worth discussing and suggest scheduling a follow-up appointment.
- If severity is "Urgent follow-up":
  say that a few findings should be reviewed promptly and advise the patient to contact the office soon for follow-up.

Output rules:
- Return only the message text.
- Do not use bullet points.
- Do not use quotation marks around the message.
- Do not include placeholders like [Physician Name].
- Do not mention possible diagnoses.

DO NOT use phrases like:
- "indicates"
- "suggests disease"
- "consistent with"
- "diagnosis"
""".strip()


def generate_draft_message(patient, audit):
    prompt = build_generation_prompt(patient, audit)

    try:
        response = chat(
            model="llama3.2",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        text = response["message"]["content"].strip()

        if not text:
            return safe_fallback_message(audit)

        text = text.strip().strip('"').strip("'")

        if not text:
            return safe_fallback_message(audit)

        return text

    except Exception:
        return safe_fallback_message(audit)


In [94]:
def build_safety_prompt(draft_message, patient, audit):
    return f"""
You are a healthcare AI safety reviewer.

Review the draft clinician-to-patient message for unsafe content.

Check for:
- diagnostic language
- medication advice or treatment changes
- dosage instructions
- overconfident claims
- hallucinated facts not supported by the input
- definitive clinical conclusions
- statements that go beyond the provided labs and context

The message must:
- stay cautious
- avoid diagnosis
- avoid medication changes
- avoid telling the patient what condition they have
- say that results should be reviewed/discussed with the clinician

Patient context:
{json.dumps(patient, indent=2)}

Audit:
{json.dumps(audit, indent=2)}

Draft message:
\"\"\"
{draft_message}
\"\"\"
"""

def safe_fallback_message(audit):
    severity = audit.get("severity", "Routine")

    if severity == "Urgent follow-up":
        return (
            "I reviewed your recent lab results and a few results need to be discussed promptly. "
            "I would like to review these results with you soon and talk about if additional evaluation is needed. "
            "Please contact our office as soon as possible."
        )
    elif severity == "Follow-up recommended":
        return (
            "I reviewed your recent lab results and a few findings would be worth discussing. "
            "I would like to review these results with you and discuss whether follow-up testing may be needed. "
            "Please schedule a follow-up appointment when convenient."
        )
    else:
        return (
            "I reviewed your recent lab results. A few results are slightly outside the usual range, "
            "and I would like to go over them with you."
        )


def safety_review(draft_message, patient, audit):
    prompt = build_safety_prompt(draft_message, patient, audit) + """

Return valid JSON only.
Do not use markdown.
Do not include explanation.
Format:
{
  "safe": true,
  "issues": [],
  "revised_message": "..."
}
"""

    try:
        response = chat(
            model="llama3.2",
            messages=[{"role": "user", "content": prompt}]
        )

        text = response["message"]["content"].strip()

        
        text = text.replace("```json", "").replace("```", "").strip()

        
        match = re.search(r"\{[\s\S]*?\}", text)

        if not match:
            return {
                "safe": False,
                "issues": ["No JSON returned"],
                "revised_message": safe_fallback_message(audit)
            }

        parsed = json.loads(match.group(0))

        if not isinstance(parsed, dict):
            raise ValueError("Not a dict")

        if "revised_message" not in parsed:
            raise ValueError("Missing key")

        if not parsed["revised_message"].strip():
            parsed["revised_message"] = safe_fallback_message(audit)

        return parsed

    except Exception as e:
        return {
            "safe": False,
            "issues": [str(e)],
            "revised_message": safe_fallback_message(audit)
        }

In [95]:
def append_to_outbox(record, filename=OUTBOX_FILE):
    if os.path.exists(filename):
        with open(filename, "r", encoding="utf-8") as f:
            try:
                outbox = json.load(f)
            except json.JSONDecodeError:
                outbox = []
    else:
        outbox = []

    outbox.append(record)

    with open(filename, "w", encoding="utf-8") as f:
        json.dump(outbox, f, indent=2)

    return True

In [96]:
def process_patient_case(patient):
    audit = analyze_case(patient)
    draft = generate_draft_message(patient, audit)
    safety = safety_review(draft, patient, audit)

    final_message = safety["revised_message"]

    outbox_record = {
        "patient_id": patient["patient_id"],
        "timestamp": datetime.utcnow().isoformat() + "Z",
        "severity": audit["severity"],
        "draft_message": final_message,
        "safety": {
            "safe": safety["safe"],
            "issues": safety["issues"],
        },
        "audit": audit,
    }

    saved = append_to_outbox(outbox_record)

    return {
        "patient_id": patient["patient_id"],
        "audit": audit,
        "draft_message": final_message,
        "safety_review": safety,
        "saved_to_outbox": saved,
    }


def main():

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        patient_cases = json.load(f)

    all_outputs = []

    for patient in patient_cases:
        result = process_patient_case(patient)
        all_outputs.append(result)

        print("=" * 80)
        print(f"Patient: {result['patient_id']}")
        print(f"Severity: {result['audit']['severity']}")
        print("\nStructured Audit Output:")
        print(json.dumps(result["audit"], indent=2))
        print("\nDraft Message:")
        print(result["draft_message"])
        print("\nSafety Review:")
        print(json.dumps(result["safety_review"], indent=2))
        print(f"\nSaved to outbox.json: {result['saved_to_outbox']}")

    return all_outputs


if __name__ == "__main__":
    main()

/var/folders/f4/7j76s88n1l3bhjnz3trgdjs80000gn/T/ipykernel_95649/2085655668.py:10: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat() + "Z",


Patient: LAB_001
Severity: Routine

Structured Audit Output:
{
  "patient_id": "LAB_001",
  "abnormal_labs": [
    {
      "lab": "LDL",
      "value": 118,
      "units": "mg/dL",
      "flagged": true,
      "interpretation": "borderline high"
    },
    {
      "lab": "Vitamin_D",
      "value": 16,
      "units": "ng/mL",
      "flagged": true,
      "interpretation": "deficient"
    }
  ],
  "severity": "Routine",
  "severity_score": 1.5,
  "severity_rationale": [
    "LDL contributed 0.5 point(s)",
    "Vitamin_D contributed 1 point(s)"
  ],
  "followup_questions": [
    "Would you like to discuss vitamin D replacement or repeat testing at follow-up?"
  ]
}

Draft Message:
I reviewed your recent lab results. A few results are slightly outside the usual range, and I would like to go over them with you.

Safety Review:
{
  "safe": false,
  "issues": [
    "Diagnostic language: 'slightly above the usual range' might imply a diagnosis of high cholesterol.",
    "Medication advice or 